In [24]:
from pathlib import Path

import pandas as pd

FOLDERS = [2, 3, 4, 5, 6, 8, 9, 10, 11, 13, 14]  # 1-14 except 1, 7 and 12

df = pd.concat(
    [pd.read_csv(f"datasets/{n}_keypoints.csv")
     for n in FOLDERS if Path(f"datasets/{n}_keypoints.csv").exists()],
    ignore_index=True,
)

In [34]:
# Step 1: features (63 keypoints) and target (single-letter A-Z)
COORD_COLS = [f"{ax}{i}" for i in range(21) for ax in ("x", "y", "z")]

data = df[df["detected"] == 1].copy()
data["label"] = data["label"].astype(str).str.upper()
data = data[data["label"].str.fullmatch(r"[A-Z]")]
data = data.dropna(subset=COORD_COLS)

X = data[COORD_COLS].values
y = data["label"].values
print(X.shape, "samples |", len(set(y)), "classes:", "".join(sorted(set(y))))

(345711, 63) samples | 26 classes: ABCDEFGHIJKLMNOPQRSTUVWXYZ


In [35]:
# Step 2: split into train / test sets
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(len(X_train), "train |", len(X_test), "test")

276568 train | 69143 test


In [ ]:
# Step 3: train the linear model (logistic regression = linear classifier for A-Z)
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=2000)
model.fit(X_train, y_train)

In [37]:
# Step 4: evaluate on the test set
from sklearn.metrics import accuracy_score

acc = accuracy_score(y_test, model.predict(X_test))
print(f"test accuracy: {acc:.3f}")

test accuracy: 0.862


In [38]:
# Step 5: the final weights (one row of 63 coefficients per letter)
weights = pd.DataFrame(model.coef_, index=model.classes_, columns=COORD_COLS)
weights["intercept"] = model.intercept_
weights

,x0,y0,z0,x1,y1,z1,x2,y2,z2,x3,...,x18,y18,z18,x19,y19,z19,x20,y20,z20,intercept
A,2.683621,3.296940,3.002334,-0.133346,4.197486,-0.928393,0.634965,-2.326375,10.311403,4.382931,...,-1.608309,-14.525840,-5.644776,1.658805,3.521290,-14.021933,2.322554,23.963820,-3.876101,0.207238
B,-0.536426,11.815624,0.741116,0.902098,12.052926,0.521793,2.341977,12.207366,-4.525478,2.132501,...,1.411930,-0.095971,3.065981,0.323532,-14.818844,3.871525,-1.935298,-34.015072,-0.062527,-4.380106
C,3.692299,-9.897078,-2.194663,-3.579905,1.924092,2.726537,-6.832636,13.204011,5.763752,-5.331381,...,2.707824,-3.323161,1.335187,-0.901122,-9.967078,-1.054429,0.176671,-15.372984,-2.779626,1.090889
D,0.089849,11.837784,5.856780,2.246953,11.493549,7.692799,7.203543,12.286901,4.205623,6.142136,...,-3.465839,-4.484445,-0.467669,1.947661,-16.877298,4.934665,4.998212,-21.356148,2.822252,-1.269304
E,-2.599358,11.800317,2.373990,0.311858,9.580176,1.153221,3.256697,11.225728,-0.366680,-1.446688,...,-3.940163,-16.138227,4.651306,-0.762269,-12.963615,-5.793393,-0.523313,10.780300,-10.662921,-2.713626
F,7.290061,3.962837,-4.671122,4.055368,1.638939,0.720316,1.859126,1.304304,2.824964,0.416618,...,-0.638290,5.849187,-0.994912,-3.025801,-3.186888,0.151823,-1.451909,-16.832737,-6.664501,-1.119600
G,0.072608,-1.351058,2.429873,3.486716,-10.499231,-0.531165,4.902753,-16.406802,1.339626,1.214609,...,-1.740619,15.866740,4.197414,2.613503,12.175907,4.721588,2.893690,12.399061,1.963527,2.348707
H,4.084691,-8.880787,-6.243466,5.304662,-20.590956,0.016636,1.749578,-19.001770,3.187364,0.934718,...,1.945421,19.076929,0.592901,6.004529,12.654601,7.461478,8.440211,6.981407,11.114473,2.712872
I,1.858997,7.445555,-3.812422,2.821847,7.599459,-2.519139,-0.001093,4.006573,-2.606455,-5.209192,...,-0.927399,-11.209283,-1.010232,-3.987495,-28.955496,3.561447,-5.296881,-48.470503,-2.844509,-0.606406
J,3.946899,1.935215,0.955952,2.632072,-15.172832,0.196877,0.263891,-20.658927,0.124525,-5.702161,...,-1.263091,19.707719,8.242381,-5.658230,0.925264,8.862523,-4.650380,-21.982431,-13.210156,3.049802


In [ ]:
# Step 6: save the trained model (weights) so app.py can load it
import joblib

joblib.dump(model, "asl_logreg.joblib")
weights.to_csv("asl_logreg_weights.csv")
print("saved asl_logreg.joblib and asl_logreg_weights.csv")